# S4 - ML Distribuido con Spark MLlib (Regresión) — base para S10

**Proyecto Sello — Equipo LLSW3.** Este cuaderno reconstruye el Data Lake **Gold** de S04 (`campo_electrico_particionado`) que S10 reutiliza, y entrena el modelo de regresión de S04.

**Datos sintéticos alineados con el simulador.** Las tres fuentes Bronze (`datos_s04/*.csv`, 100 000 lecturas, **una por segundo**) se generaron con `datos_s04/generar_datos_sinteticos.py`, que usa el **mismo módulo** que el productor Kafka de S10 (`uso-campo-electrico/app/simulador_campo_electrico.py`): mismos rangos, mismo paso máximo y misma dinámica de `CE`. Así, lo que el modelo aprende del histórico es exactamente el proceso que luego llega en vivo por Kafka.

| Capa | Artefacto |
|---|---|
| Bronze | `datos_s04/campo_electrico.csv`, `campo_magnetico.csv`, `variables_ambientales.csv` |
| Silver | `df_valido`: integrado por `DateTime`, deduplicado, sin `CM = 99999`, sin nulos |
| Gold | `artifacts/s04/campo_electrico_particionado/` (Parquet particionado por `Fecha`) |

## 3.1 Fase 1 — Business Understanding

In [1]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
ORIGEN_DATOS = "/opt/datos_s04"
ARTIFACTS = "/opt/artifacts/s04"
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/21 14:52:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0


**Objetivo:** estimar `CE` (campo eléctrico) en un instante a partir de las otras 7 variables físicas medidas en ese mismo instante (`CM`, `TempOut`, `OutHum`, `WindSpeed`, `Bar`, `SolarRad`, `UVIndex`). Se comparan configuraciones de regresión lineal y un Random Forest con RMSE, R² y MAE.

## 3.2 Fase 2 — Data Understanding

### 3.2.1 Cargar las tres fuentes con esquema explícito

In [2]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType
schema_ce = StructType([StructField("DateTime", TimestampType(), False), StructField("CE", DoubleType(), True)])
schema_cm = StructType([StructField("DateTime", TimestampType(), False), StructField("CM", DoubleType(), True)])
schema_va = StructType([
    StructField("TempOut", DoubleType(), True), StructField("OutHum", DoubleType(), True),
    StructField("WindSpeed", DoubleType(), True), StructField("Bar", DoubleType(), True),
    StructField("SolarRad", DoubleType(), True),   # en el CSV la cabecera es "SolarRad." (con punto)
    StructField("UVIndex", DoubleType(), True), StructField("DateTime", TimestampType(), False),
])
df_ce = spark.read.csv(f"{ORIGEN_DATOS}/campo_electrico.csv", header=True, schema=schema_ce)
df_cm = spark.read.csv(f"{ORIGEN_DATOS}/campo_magnetico.csv", header=True, schema=schema_cm)
df_va = spark.read.csv(f"{ORIGEN_DATOS}/variables_ambientales.csv", header=True, schema=schema_va)
for nombre, d in [("campo_electrico", df_ce), ("campo_magnetico", df_cm), ("variables_ambientales", df_va)]:
    print(f"{nombre:22s} {d.count():>7,} filas | DateTime distintos: {d.select('DateTime').distinct().count():,}")

campo_electrico        100,000 filas | DateTime distintos: 100,000


campo_magnetico        100,000 filas | DateTime distintos: 100,000


variables_ambientales  101,200 filas | DateTime distintos: 100,000


`variables_ambientales` trae 1 200 segundos exportados dos veces (una copia incompleta). Se deduplica **antes** del `join`, conservando la fila con menos nulos:

In [3]:
from pyspark.sql.functions import col, when, row_number, count as spark_count
from pyspark.sql.window import Window
cols_va = [c for c in df_va.columns if c != "DateTime"]
df_va_unico = (
    df_va.withColumn("CantidadNulos", sum(when(col(c).isNull(), 1).otherwise(0) for c in cols_va))
    .withColumn("row_num", row_number().over(Window.partitionBy("DateTime").orderBy(col("CantidadNulos").asc())))
    .filter(col("row_num") == 1).drop("row_num", "CantidadNulos")
)
print(f"variables_ambientales deduplicado: {df_va_unico.count():,} filas")
df_integrado = df_ce.join(df_cm, on="DateTime", how="left").join(df_va_unico, on="DateTime", how="left")
print(f"Integrado: {df_integrado.count():,} registros x {len(df_integrado.columns)} columnas")

variables_ambientales deduplicado: 100,000 filas


Integrado: 100,000 registros x 9 columnas


### 3.2.2 - 3.2.4 Esquema, nulos, estadísticas y correlación con `CE`

In [4]:
df_integrado.printSchema()
df_integrado.select([spark_count(when(col(c).isNull(), c)).alias(c) for c in df_integrado.columns]).show()
VARIABLES_MODELO = ["CE", "CM", "TempOut", "OutHum", "WindSpeed", "Bar", "SolarRad", "UVIndex"]
df_integrado.describe(VARIABLES_MODELO).show()
for c in VARIABLES_MODELO[1:]:
    print(f"{c:10s} correlación con CE: {df_integrado.stat.corr(c, 'CE'): .4f}")

root
 |-- DateTime: timestamp (nullable = true)
 |-- CE: double (nullable = true)
 |-- CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)



+--------+---+---+-------+------+---------+---+--------+-------+
|DateTime| CE| CM|TempOut|OutHum|WindSpeed|Bar|SolarRad|UVIndex|
+--------+---+---+-------+------+---------+---+--------+-------+
|       0|  0|  0|      0|     0|        0|  0|       0|      0|
+--------+---+---+-------+------+---------+---+--------+-------+



+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|                 CE|                CM|           TempOut|           OutHum|        WindSpeed|               Bar|          SolarRad|           UVIndex|
+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|             100000|            100000|            100000|           100000|           100000|            100000|            100000|            100000|
|   mean|-1.6223409000000066|24274.781643900165| 21.90588159999961|77.97553089999944| 9.23213059999987| 949.4709016000066| 377.1388645000001|  2.76970449999997|
| stddev| 1.0200210672417387|3216.6096512753693|3.5093855306377475| 8.50035358713156|4.955453450062073|2.2113073865750184|238.56101578065952|1.7439529676569019|
|    min|              -4.91|     

CM         correlación con CE:  0.0015


TempOut    correlación con CE:  0.6380


OutHum     correlación con CE: -0.1007


WindSpeed  correlación con CE: -0.6899


Bar        correlación con CE:  0.0010


SolarRad   correlación con CE:  0.3346


UVIndex    correlación con CE:  0.0064


`CM` todavía incluye el código de error `99999`, que distorsiona su estadística (máximo y desviación). Se limpia en la Fase 3.

## 3.3 Fase 3 — Data Preparation

### 3.3.1 - 3.3.2 Código de error, duplicados y nulos (Silver)

In [5]:
errores_cm = df_integrado.filter(col("CM") == 99999).count()
df_limpio = df_integrado.filter(col("CM") != 99999)
print(f"Filas con CM=99999: {errores_cm:,} -> quedan {df_limpio.count():,}")
total, sin_dup = df_limpio.count(), df_limpio.dropDuplicates(["DateTime"]).count()
assert total == sin_dup, "Hay DateTime duplicada"
df_valido = df_limpio.na.drop(subset=VARIABLES_MODELO).cache()
print(f"Sin DateTime duplicada: {sin_dup:,} | después de na.drop: {df_valido.count():,}")

Filas con CM=99999: 180 -> quedan 99,820


Sin DateTime duplicada: 99,820 | después de na.drop: 99,820


### Escribir la capa Gold particionada por `Fecha`

In [6]:
from pyspark.sql.functions import date_format
import os
RUTA_GOLD = f"{ARTIFACTS}/campo_electrico_particionado"
(df_valido.withColumn("Fecha", date_format(col("DateTime"), "yyyy-MM-dd"))
    .repartition(4).write.format("parquet").mode("overwrite").partitionBy("Fecha").save(RUTA_GOLD))
print(sorted(os.listdir(RUTA_GOLD)))
df = spark.read.parquet(RUTA_GOLD)
assert df.count() == df_valido.count()
print(f"Gold verificado: {df.count():,} filas, ida y vuelta sin pérdida")
df.groupBy("Fecha").count().orderBy("Fecha").show()
df.filter(col("Fecha") == "2026-10-20").explain()

['._SUCCESS.crc', 'Fecha=2026-10-19', 'Fecha=2026-10-20', '_SUCCESS']


Gold verificado: 99,820 filas, ida y vuelta sin pérdida


+----------+-----+
|     Fecha|count|
+----------+-----+
|2026-10-19|86246|
|2026-10-20|13574|
+----------+-----+

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [DateTime#3343,CE#3344,CM#3345,TempOut#3346,OutHum#3347,WindSpeed#3348,Bar#3349,SolarRad#3350,UVIndex#3351,Fecha#3352] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/artifacts/s04/campo_electrico_particionado], PartitionFilters: [isnotnull(Fecha#3352), (Fecha#3352 = 2026-10-20)], PushedFilters: [], ReadSchema: struct<DateTime:timestamp,CE:double,CM:double,TempOut:double,OutHum:double,WindSpeed:double,Bar:d...




El plan muestra `PartitionFilters: [isnotnull(Fecha), (Fecha = 2026-10-20)]`: al filtrar por la columna de partición, Spark solo lee esa carpeta.

### 3.3.3 - 3.3.5 Predictores, `VectorAssembler` y división aleatoria (válida en S04: cada fila es una foto independiente)

In [7]:
from pyspark.ml.feature import VectorAssembler
PREDICTORES = [v for v in VARIABLES_MODELO if v != "CE"]
df_ml = VectorAssembler(inputCols=PREDICTORES, outputCol="features").transform(df).select("features", "CE")
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Predictores: {PREDICTORES}")
print(f"Entrenamiento: {df_train.count():,} | Prueba: {df_test.count():,}")

Predictores: ['CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'SolarRad', 'UVIndex']


Entrenamiento: 80,017 | Prueba: 19,803


## 3.4 Fase 4 — Modeling

### 3.4.1 - 3.4.3 Regresión lineal y regularización

In [8]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
def evaluar(pred, nombre):
    r = {m.upper(): RegressionEvaluator(labelCol="CE", predictionCol="prediction", metricName=m).evaluate(pred) for m in ["rmse", "r2", "mae"]}
    r["Configuracion"] = nombre
    print(f"{nombre:22s} RMSE={r['RMSE']:.4f}  R2={r['R2']:.4f}  MAE={r['MAE']:.4f}")
    return r
resultados, modelos = [], {}
for nombre, reg, en in [("Sin regularizacion", 0.0, 0.0), ("Ridge (L2)", 0.1, 0.0), ("Elastic Net (L1+L2)", 0.1, 0.5)]:
    m = LinearRegression(featuresCol="features", labelCol="CE", regParam=reg, elasticNetParam=en).fit(df_train)
    modelos[nombre] = m
    resultados.append(evaluar(m.transform(df_test), nombre))
base = modelos["Sin regularizacion"]
print("\nCoeficientes (", ", ".join(PREDICTORES), "):", [round(float(c), 4) for c in base.coefficients], "Intercepto:", round(base.intercept, 4))

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Sin regularizacion     RMSE=0.1671  R2=0.9729  MAE=0.1329


Ridge (L2)             RMSE=0.1913  R2=0.9646  MAE=0.1518


Elastic Net (L1+L2)    RMSE=0.2235  R2=0.9516  MAE=0.1786

Coeficientes ( CM, TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex ): [-0.0, 0.1806, -0.0293, -0.1185, 0.0011, 0.002, 0.0025] Intercepto: -2.8698


### 3.4.4 - 3.4.6 Random Forest e importancia de variables

In [9]:
modelo_rf = RandomForestRegressor(featuresCol="features", labelCol="CE", numTrees=50, maxDepth=8, seed=42).fit(df_train)
modelos["Random Forest"] = modelo_rf
resultados.append(evaluar(modelo_rf.transform(df_test), "Random Forest"))
for v, imp in sorted(zip(PREDICTORES, modelo_rf.featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{v:10s} {imp:.4f}")

Random Forest          RMSE=0.1865  R2=0.9663  MAE=0.1479
WindSpeed  0.3993
TempOut    0.3354
SolarRad   0.1504
OutHum     0.0415
CM         0.0363
Bar        0.0245
UVIndex    0.0127


## 3.5 Fase 5 — Evaluation y 3.6 Deployment

In [10]:
import pandas as pd
tabla = pd.DataFrame(resultados)[["Configuracion", "RMSE", "R2", "MAE"]].sort_values("RMSE")
print(tabla.to_string(index=False))
ganador = tabla.iloc[0]["Configuracion"]
RUTA_MODELO_S04 = f"{ARTIFACTS}/modelo_ce_regresion"
modelos[ganador].write().overwrite().save(RUTA_MODELO_S04)
print(f"\nGanador: {ganador} -> guardado en {RUTA_MODELO_S04}")
_ = df_valido.unpersist()

      Configuracion     RMSE       R2      MAE
 Sin regularizacion 0.167139 0.972943 0.132928
      Random Forest 0.186468 0.966323 0.147945
         Ridge (L2) 0.191290 0.964559 0.151844
Elastic Net (L1+L2) 0.223549 0.951597 0.178609



Ganador: Sin regularizacion -> guardado en /opt/artifacts/s04/modelo_ce_regresion


**Lectura:** la dinámica del simulador hace que `CE` tienda hacia un nivel que depende de `TempOut` (+), `WindSpeed` (−) y `OutHum` (−), y la regresión lineal lo recupera con signos coherentes. Esta misma capa Gold (`campo_electrico_particionado`) es la entrada de S10, donde la pregunta cambia: ya no qué explica `CE` en el mismo instante, sino si el presente predice el **segundo siguiente**.